In [11]:
import numpy as np

def euler_zyx_to_R(u_deg, v_deg, w_deg):
    """
    Roll(u, X), Pitch(v, Y), Yaw(w, Z) [deg] ->
    Rotation matrix R = Rz(w) * Ry(v) * Rx(u)
    """
    u = np.deg2rad(u_deg)
    v = np.deg2rad(v_deg)
    w = np.deg2rad(w_deg)

    Rx = np.array([
        [1,          0,           0        ],
        [0,  np.cos(u),  -np.sin(u)],
        [0,  np.sin(u),   np.cos(u)]
    ])

    Ry = np.array([
        [ np.cos(v), 0, np.sin(v)],
        [ 0,         1, 0        ],
        [-np.sin(v), 0, np.cos(v)]
    ])

    Rz = np.array([
        [np.cos(w), -np.sin(w), 0],
        [np.sin(w),  np.cos(w), 0],
        [0,          0,         1]
    ])

    R = Rz @ Ry @ Rx
    return R


def compute_tip_R_and_pose(tip_p, tip_length, u_deg, v_deg, w_deg,
                           flange_pos=None, flange_R=None):
    """
    입력:
        tip_p      : 플랜지 좌표계에서 툴 기준점까지 오프셋 [px, py, pz]
        tip_length : 툴 기준점 -> 툴 끝(TCP)까지 길이 (스칼라)
        u_deg, v_deg, w_deg : tip_R_를 정의하는 Euler 각 (deg)
        flange_pos : (옵션) 월드에서 플랜지 위치 [fx, fy, fz]
                     None이면 [0,0,0] 으로 가정 (플랜지 좌표 = 월드 좌표)
        flange_R   : (옵션) 월드에서 플랜지 회전(3x3)
                     None이면 단위행렬로 가정

    출력:
        tip_R_ : 3x3 회전행렬 (툴 frame)
        pose   : 6D 벡터 [x, y, z, u, v, w]  (여기서 x,y,z는 flange/world 기준)
    """
    tip_p = np.asarray(tip_p, dtype=float).reshape(3)
    if flange_pos is None:
        flange_pos = np.zeros(3)
    else:
        flange_pos = np.asarray(flange_pos, dtype=float).reshape(3)

    if flange_R is None:
        flange_R = np.eye(3)
    else:
        flange_R = np.asarray(flange_R, dtype=float).reshape(3, 3)

    # 1) u, v, w -> tip_R_
    tip_R = euler_zyx_to_R(u_deg, v_deg, w_deg)

    # 2) 툴 방향 (world 기준) : R_flange * tip_R * ez
    ez = np.array([0.0, 0.0, 1.0])
    tool_dir_world = flange_R @ (tip_R @ ez)

    # 3) TCP 위치: 플랜지 + 플랜지→툴기준점 + 툴방향 * tip_length
    tcp_pos_world = flange_pos + flange_R @ tip_p + tip_length * tool_dir_world

    pose = np.array([
        tcp_pos_world[0],
        tcp_pos_world[1],
        tcp_pos_world[2],
        float(u_deg),
        float(v_deg),
        float(w_deg),
    ])

    return tip_R, pose




In [ ]:
# 예시 사용
if __name__ == "__main__":
    tip_p = [0.0, 0.0, 0.11243]
    tip_length = 0.1032234
    u, v, w = 30.0, -35.2643897, 0.0

    tip_R_, pose = compute_tip_R_and_pose(tip_p, tip_length, u, v, w)

    np.set_printoptions(precision=6, suppress=True)
    print("tip_R_ =\n", tip_R_)
    pose[0] = pose[0] * 1000    
    pose[1] = pose[1] * 1000    
    pose[2] = pose[2] * 1000    
    print("pose [x, y, z, u, v, w] =\n", pose)

# 적용 순서(물리적 회전):   Roll(u) → Pitch(v) → Yaw(w)
# 행렬 곱 순서:             R = Rz(w) · Ry(v) · Rx(u)
# Euler 타입 이름:          ZYX Euler


tip_R_ =
 [[ 0.816497 -0.288675 -0.5     ]
 [ 0.        0.866025 -0.5     ]
 [ 0.57735   0.408248  0.707107]]
pose [x, y, z, u, v, w] =
 [-51.6117   -51.6117   185.419966  30.       -35.26439    0.      ]
